<a href="https://colab.research.google.com/github/j019/Practical-Machine-Learning/blob/main/Day17/Recommendation_itemtrust.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "numpy<2.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-pyth

In [1]:
!pip install surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.1 MB/s eta 0:00:00


In [2]:
import surprise

In [3]:
import pandas as pd
import numpy as np
import surprise

In [4]:
ratings = pd.read_csv("/content/item_ratings.txt",sep=' ',names = ['uid','iid','rating'])
ratings.head()

,uid,iid,rating
0,1,1,2.0
1,1,2,4.0
2,1,3,3.5
3,1,4,3.0
4,1,5,4.0


In [5]:
lowest_rating = ratings['rating'].min()
highest_rating = ratings['rating'].max()

print("Ratings range between {0} and {1}".format(lowest_rating,highest_rating))

Ratings range between 0.5 and 4.0


In [6]:
reader = surprise.Reader(rating_scale = (lowest_rating,highest_rating))
data = surprise.Dataset.load_from_df(ratings,reader)

In [7]:
# AutoFolds --> We are keeping huge data in Folds
type(data)

surprise.dataset.DatasetAutoFolds

# Train the Algorithm for User Based Rating Prediction

In [98]:
similarity_options = {'name': 'cosine',
                      'user_based': True}
# Default k = 40
algo = surprise.KNNBasic(min_k=2,
                         sim_options = similarity_options,
                          verbose=5)
output = algo.fit(data.build_full_trainset())

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [99]:
algo.__dict__.keys()

dict_keys(['bsl_options', 'sim_options', 'verbose', 'k', 'min_k', 'trainset', 'bu', 'bi', 'n_x', 'n_y', 'xr', 'yr', 'sim'])

In [100]:
algo.sim.shape

(1508, 1508)

In [101]:
ratings.nunique()

,0
uid,1508
iid,2071
rating,8


# Predict the rating of a Single Item by an User

In [102]:
pred = algo.predict(uid='708',iid='1507')
score = pred.est
print(score)

3.0028030537791928


### Verify from original Data

In [103]:
ratings.loc[(ratings['uid'] == 708) & (ratings['iid'] == 1507) , :]

,uid,iid,rating
17178,708,1507,1.5


In [104]:
pred = algo.predict(uid='50',iid='6')
score = pred.est
print(score)

3.0028030537791928


In [105]:
ratings.loc[(ratings['uid'] == 50) & (ratings['iid'] == 6) , :]

,uid,iid,rating
884,50,6,4.0


# Create Recommendations for an User id 707

## Predict Ratings of all Unrated Items by an User id 707

In [106]:
iids = ratings['iid'].unique()
iids50 = ratings.loc[ratings['uid'] == 707 ,'iid']
print("List of iid that uid={0} has rated:".format(707))
print(iids50)

List of iid that uid=707 has rated:
17170     10
17171    235
17172      2
17173    341
17174    207
17175      7
Name: iid, dtype: int64


In [107]:
iids_to_predict = np.setdiff1d(iids,iids50)
print("List of iid which uid={0} did not rate(in all {1}) :".format(707,len(iids_to_predict)))
print(iids_to_predict)
print("no of unrated items", len(iids_to_predict))

List of iid which uid=707 did not rate(in all 2065) :
[   1    3    4 ... 2069 2070 2071]
no of unrated items 2065


In [108]:
### ratings arbitrarily set to 0
testset = [[707,iid,0.] for iid in iids_to_predict]
predictions = algo.test(testset)
predictions[:-5]

[Prediction(uid=707, iid=1, r_ui=0.0, est=3.125, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid=707, iid=3, r_ui=0.0, est=3.0125, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid=707, iid=4, r_ui=0.0, est=3.424939968286814, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid=707, iid=5, r_ui=0.0, est=3.374996485072647, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid=707, iid=6, r_ui=0.0, est=3.087560862979787, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid=707, iid=8, r_ui=0.0, est=3.0000099088876437, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid=707, iid=9, r_ui=0.0, est=3.3500094216111687, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid=707, iid=11, r_ui=0.0, est=3.575, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid=707, iid=12, r_ui=0.0, est=2.687494494150231, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(ui

In [109]:
pred_ratings = np.array([pred.est for pred in predictions])
pred_ratings[:5]

array([3.125     , 3.0125    , 3.42493997, 3.37499649, 3.08756086])

In [110]:
pred_ratings.shape

(2065,)

In [111]:
# Finding the index of maximum predicted rating
i_max = pred_ratings.argmax()

In [112]:
# Recommending the item with maximum predicted rating
iid_recommend_most = iids_to_predict[i_max]
print("Top item to be recommended for user {0} is {1} with predicted rating as {2}".format(707,iid_recommend_most,pred_ratings[i_max]))

Top item to be recommended for user 707 is 35 with predicted rating as 4.0


In [113]:
# Getting top 10 items to be recommended for uid = 707
import heapq # use priority queue algorithm.
i_sorted_10 = heapq.nlargest(10, range(len(pred_ratings)), pred_ratings.take)
top_10_items = iids_to_predict[i_sorted_10]
print(top_10_items)

[ 35  54  68  97 107 111 118 136 142 162]


In [97]:
# for uid = 707 'msd similarity'
# [ 35  54  68  97 107 111 118 136 142 162]

In [80]:
# for uid = 707 'cosine similarity'
#[ 23  33  74  79 101 109 111 133 138 140] --> Item Based
#[ 11  35  54  68  97 107 111 118 136 142] --> User Based

In [ ]:
# Task Uid = 700 / 707 / 708

# Check performance of Recommendation system

# Train test split

In [114]:
from surprise.model_selection import train_test_split

In [115]:
X_train, X_test = train_test_split(data,
                                   test_size=0.3,
                                   random_state=7)

In [116]:
X_train

In [258]:
similarity_options = {'name': 'pearson_baseline',
                      'user_based': False}
# Default k = 40
# k = 5, 40, 100
# min_k = 2, 5
algo = surprise.KNNBasic(min_k = 2,k = 40,sim_options = similarity_options,
                         verbose=5)
output = algo.fit(X_train)

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.


In [259]:
predictions = algo.test(X_test)

In [260]:
from surprise.accuracy import mae , mse

In [261]:
mae(predictions)

MAE:  0.6266


0.6265790984511914

In [262]:
mse(predictions)

MSE: 0.6858


0.6857738912645619

In [263]:
len(X_test)

10650

# Check Count of Impossible Predictions

In [264]:
count = 0
for p in predictions:
  if p.details['was_impossible']:
    count += 1
print(count)

1415


In [265]:
predictions[10000:10050]

[Prediction(uid=105, iid=254, r_ui=2.5, est=3.005413128345474, details={'was_impossible': True, 'reason': 'Not enough neighbors.'}),
 Prediction(uid=1101, iid=214, r_ui=2.5, est=3.0518257614496886, details={'actual_k': 5, 'was_impossible': False}),
 Prediction(uid=891, iid=2, r_ui=2.5, est=2.404590589474929, details={'actual_k': 10, 'was_impossible': False}),
 Prediction(uid=825, iid=728, r_ui=4.0, est=3.8290511236415297, details={'actual_k': 8, 'was_impossible': False}),
 Prediction(uid=3, iid=148, r_ui=3.5, est=2.844399153447355, details={'actual_k': 4, 'was_impossible': False}),
 Prediction(uid=1417, iid=419, r_ui=4.0, est=3.3797076634455303, details={'actual_k': 6, 'was_impossible': False}),
 Prediction(uid=1355, iid=593, r_ui=4.0, est=2.5278132220526386, details={'actual_k': 6, 'was_impossible': False}),
 Prediction(uid=209, iid=245, r_ui=3.0, est=2.924874330327155, details={'actual_k': 20, 'was_impossible': False}),
 Prediction(uid=72, iid=235, r_ui=3.0, est=3.4084853872440277, d